## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [2]:
import os
import pandas as pd
from openai import OpenAI
import json

# Instantiate an API client
client = OpenAI()

# Step 1: Build nasdaq100_ca with ytd column
nasdaq100_ca = pd.read_csv("nasdaq100_CA.csv")
price_change = pd.read_csv("nasdaq100_price_change.csv")

# Step 2: Merging only the ytd column & not every price column using the shared 'symbol' column
nasdaq100_ca = nasdaq100_ca.merge(
    price_change[["symbol", "ytd"]],
    on="symbol",
    how="left"
)


#Step 3: Save nasdaq_ytd.csv
nasdaq100_ca.to_csv("nasdaq_ytd.csv", index=False)

#Step 4: create function to classify the sector
#def classify_sector(company_name):
#    response = client.chat.completions.create(
#        model="gpt-4o-mini",
#        messages=[{"role": "user", "content": (f"Classify {company_name} into exactly one of these sectors: Technology, Consumer Cyclical, Industrials, Utilities, #Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial. Reply with only the sector name, nothing else.")}
#                 ]
#    )
#    return response.choices[0].message.content.strip()
#
#nasdaq100_ca["sector"] = nasdaq100_ca["name"].apply(classify_sector)

def classify_all_sectors(df):
    company_list = "\n".join(
        f"{row.symbol}: {row.name}" for row in df.itertuples()
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": (
                "Classify each company below into exactly one of these sectors: "
                "Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, "
                "Communication, Energy, Consumer Defensive, Real Estate, Financial.\n\n"
                "Return ONLY a JSON array like this:\n"
                '[{"symbol": "AAPL", "sector": "Technology"}, ...]\n\n'
                "No extra text, no markdown, just the JSON array.\n\n"
                f"Companies:\n{company_list}"
            )
        }]
    )
    results = json.loads(response.choices[0].message.content)
    sector_map = {item["symbol"]: item["sector"] for item in results}
    return df["symbol"].map(sector_map)

nasdaq100_ca["sector"] = classify_all_sectors(nasdaq100_ca)


#Step 5: Generate stock_recommendations
summary = nasdaq100_ca[["symbol", "name", "sector", "ytd"]].to_json(orient="records")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": (
            "You are a senior equity analyst. Based on these Nasdaq 100 companies "
            f"with their sectors and YTD performance:\n{summary}\n\n"
            "Recommend the TWO best sectors to invest in right now, with at least "
            "two specific company picks per sector and a brief rationale for each."
        )
    }]
)

stock_recommendations = response.choices[0].message.content
print(stock_recommendations)




Based on the year-to-date (YTD) performance data provided for companies in the Nasdaq 100, the two best sectors to invest in right now are **Technology** and **Communication**. Here’s a brief rationale for each sector along with specific company picks:

### 1. Technology Sector
**Rationale:**
The Technology sector has shown remarkable YTD performance overall, with multiple companies boasting double-digit returns and some, like Nvidia, experiencing extraordinary gains over 200%. This sector's strength is largely driven by advancements in artificial intelligence, cloud computing, and strong consumer demand for innovative products and services.

**Top Picks:**
- **Nvidia (NVDA)**: YTD performance of 217.27% makes Nvidia a standout within the sector. As a leader in graphics processing units (GPUs) and AI technology, Nvidia is poised for continued growth given the increasing demand for AI applications, gaming, and data centers.
  
- **Adobe Inc. (ADBE)**: YTD performance of 57.23% suggests 